In [ ]:
#####################################
# This creates figure
# TraxAceBar.png 
# in paper
#####################################
import sys
# import modules in other directories
sys.path.append('../Utils/')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature



import trax_util as trx
import ibtracs as IBT
import parse_tempest_file as TM

import importlib
import PlotUtil as Pu

importlib.reload( TM )
importlib.reload( Pu )
importlib.reload( trx )
importlib.reload( IBT )


In [ ]:


##columns = ['cell_id','lon','lat','slp','wind','psrf','year','month','day','hour',]

#f='/glade/campaign/cgd/amp/patc/MAAG/TC_tracking/Gen_TCtracks/Maria/Results/traj.ne30x8_MARIA_NDG40_01.txt'
f='/glade/campaign/cgd/amp/patc/MAAG/TC_tracking/Gen_TCtracks/AUG_2020/Results/traj.ne30x8_NDG40_Aug2020_01.txt'
f='/glade/campaign/cgd/amp/patc/MAAG/TC_tracking/Gen_TCtracks/AUG_2020/Results/traj.ne30x4_NDG40_Aug2020_01.txt'
f='/glade/campaign/cgd/amp/patc/MAAG/TC_tracking/Gen_TCtracks/2020/Results/traj.ne30x8_2020sim_01.txt'
tcMAAG= TM.parse_storm_file( f )


In [ ]:
#plt.plot(tc2004[1]['wind'])
# Collect all wind data across storms
all_winds = []

for storm in tcMAAG:
    all_winds.extend(storm['wind'])  # Append all wind values from the current storm

# List to store the highest wind value from each storm
highest_winds = []

for storm in tcMAAG:
    if storm['wind']:  # Check if the wind list is not empty
        highest_winds.append(max(storm['wind']))


plt.plot(highest_winds)

In [ ]:
#f=trx.rcp85fname(ens=1,sst='sst3')
#f=trx.pdfname(ens=1)
power_wind=(1./6.)**(0.11)

#pd1=trx.readtrx( trx.pdfname(ens=1) , power_wind=power_wind )
#pd2=trx.readtrx( trx.pdfname(ens=2) , power_wind=power_wind )
#pd3=trx.readtrx( trx.pdfname(ens=3) , power_wind=power_wind )
#ibx=IBT.readtrx(Fill_NI_before_1990=False)
ibx=IBT.readtrx()



In [ ]:
#Maria
yook = np.where( ( ibx.year[:,0] >= 2017 )&( ibx.year[:,0] <= 2017 )& (ibx.month[:,0] == 9) & (ibx.day[:,0] >1 ) )
#Aug-Sep 2020
yook = np.where( ( ibx.year[:,0] == 2020 ) &  (ibx.month[:,0] >= 7) &  (ibx.month[:,0] <= 9) )
#All of 2020
yook = np.where( ( ibx.year[:,0] == 2020 ) )
len(yook[0])

In [ ]:
from matplotlib.legend_handler import HandlerPathCollection

central_longitude=180.

#projPC = ccrs.Robinson(central_longitude=180.)
projPC = ccrs.PlateCarree(central_longitude=central_longitude) #180.)
data_crs = ccrs.PlateCarree()

basi = IBT.basin_outlines(Mediterranean=False,xNEPacific=True)
ipan=-1
"""
Add axes method 
ax = fig.add_axes([xmin , ymin , dx , dy , ])
"""
# Color mapping for categories
categories = np.arange(-2, 6)  # -2 (sub-category storms) to 7 (Cat5)
colors = ['gray', 'blue', 'cyan', 'green', 'yellow', 'orange', 'purple', 'red']  # Red for Cat5+

Axes = np.asarray( [ 
    [ .05, .05, .8, .8 ] ,
    [ .55, .6, .4, .25] ,
    [ .05, .3, .4, .25 ] ,
    [ .55, .3, .4, .25 ] ,
    [ .3, 0., .4, .25] 
    ])

print(np.shape(Axes))

print(Axes[0,:])

scale_fac = 1e-4 * 1.944**2 
denslv=21 #np.linspace(0,15,num=16)
#denslv=np.linspace(500,10_500,num=21)
denslv=np.linspace(0.2,4.0,num=20)

#scale_fac = 1.0 #1e-4 * 1.944**2 
#denslv= np.linspace(0.002,0.052,num=21) # - 0.05


fig = plt.figure(figsize=(20, 15))
levs=[29,26,20,10]
    
ax1 = fig.add_axes( Axes[0,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
ax1.set_title( "IBTrACS Aug-Sep 2017" , fontsize=16)
ax1.add_feature(cfeature.LAND, facecolor='tan') 
###
# Plot IBTrACS basins
###
"""
bbox=dict(facecolor='white', edgecolor='black')
for b in np.arange( start=1, stop=len(basi) ):
    xyb=np.array( basi[b].xy )
    loos='-'
    if ((basi[b].number == 1) or (basi[b].number == 7)) :
        loos='--'
    ax1.plot( xyb[:,0],xyb[:,1] ,color='red',transform=data_crs,linestyle=loos)
    if (basi[b].labloc[0]>-9999):
        ax1.text( basi[b].labloc[0],basi[b].labloc[1],basi[b].name , transform=data_crs, color='red' , bbox=bbox ) 
"""
ist=0
categories =[-2,0,1,2,3,4,5]  # -2 (sub-category storms) to 7 (Cat5)
cat_labels=['TD','TS','Cat 1','Cat 2','Cat 3','Cat 4','Cat 5',]
colors = ['beige','khaki','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['green','green','gold','orange', 'darkorange','red','red']  # Red for Cat5+

for ii in np.arange(len(yook[0])):
    ist=yook[0][ii]
    coo  = trx.categorize_storms(ibx.wind[ist, :], wind=True) 
    #ax1.scatter( ibx.lon[ist,:],ibx.lat[ist,:],transform=data_crs, s=.2 )

    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        mask = (coo == cat)
        ax1.scatter(ibx.lon[ist, mask], ibx.lat[ist, mask],
                    transform=data_crs, s=0.4, color=color, label=f"{catlab}" if ii == 0 else "")
handles, labels = ax1.get_legend_handles_labels()
legend=ax1.legend(
    handles, 
    labels, 
    handler_map={plt.scatter: HandlerPathCollection()},
    loc='upper left', 
    title="Categories", 
    fontsize=14,  # Text size
    title_fontsize=16  # Title size
)
#for handle in legend.legendHandles:
for handle in legend.legend_handles:
    handle.set_sizes([100])  # Adjust this value for desired marker size



In [ ]:



ist=0
categories =[-2,0,1,2,3,4,5]  # -2 (sub-category storms) to 7 (Cat5)
cat_labels=['TD','TS','Cat 1','Cat 2','Cat 3','Cat 4','Cat 5',]
colors = ['beige','khaki','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['green','green','gold','orange', 'darkorange','red','red']  # Red for Cat5+




print( len(tcMAAG))
ist=1
wind=np.array( tcMAAG[ist]['wind'] )
#coo  = trx.categorize_storms(tcMAAG[ist]['wind'], wind=True) 
coo  = trx.categorize_storms(wind, wind=True) 





#plt.plot( wind )
#plt.plot( coo )

In [ ]:
for ist in np.arange(len(tcMAAG)):
    windtc = np.array( tcMAAG[ist]['wind'] )
    lontc  = np.array( tcMAAG[ist]['lon'] )
    lattc  = np.array( tcMAAG[ist]['lat'] )
    coo  = trx.categorize_storms(  windtc , wind=True) 
    #ax1.scatter( ibx.lon[ist,:],ibx.lat[ist,:],transform=data_crs, s=.2 )

    """
    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        mask = (coo == cat)
        ax1.scatter( lontc  , lattc  ,
                    transform=data_crs, s=0.4, color=color, label=f"{catlab}" if ii == 0 else "")
    """
    plt.scatter( lontc  , lattc  )


In [ ]:
from matplotlib.legend_handler import HandlerPathCollection

central_longitude=180.

#projPC = ccrs.Robinson(central_longitude=180.)
projPC = ccrs.PlateCarree(central_longitude=central_longitude) #180.)
data_crs = ccrs.PlateCarree()

basi = IBT.basin_outlines(Mediterranean=False,xNEPacific=True)
ipan=-1
"""
Add axes method 
ax = fig.add_axes([xmin , ymin , dx , dy , ])
"""
# Color mapping for categories
categories = np.arange(-2, 6)  # -2 (sub-category storms) to 7 (Cat5)
colors = ['gray', 'blue', 'cyan', 'green', 'yellow', 'orange', 'purple', 'red']  # Red for Cat5+

Axes = np.asarray( [ 
    [ .05, .05, .8, .8 ] ,
    [ .55, .6, .4, .25] ,
    [ .05, .3, .4, .25 ] ,
    [ .55, .3, .4, .25 ] ,
    [ .3, 0., .4, .25] 
    ])

print(np.shape(Axes))

print(Axes[0,:])

scale_fac = 1e-4 * 1.944**2 
denslv=21 #np.linspace(0,15,num=16)
#denslv=np.linspace(500,10_500,num=21)
denslv=np.linspace(0.2,4.0,num=20)

#scale_fac = 1.0 #1e-4 * 1.944**2 
#denslv= np.linspace(0.002,0.052,num=21) # - 0.05


fig = plt.figure(figsize=(20, 15))
levs=[29,26,20,10]
    
ax1 = fig.add_axes( Axes[0,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
ax1.set_title( "IBTrACS Aug-Sep 2017" , fontsize=16)
ax1.add_feature(cfeature.LAND, facecolor='tan') 
###
# Plot IBTrACS basins
###
"""
bbox=dict(facecolor='white', edgecolor='black')
for b in np.arange( start=1, stop=len(basi) ):
    xyb=np.array( basi[b].xy )
    loos='-'
    if ((basi[b].number == 1) or (basi[b].number == 7)) :
        loos='--'
    ax1.plot( xyb[:,0],xyb[:,1] ,color='red',transform=data_crs,linestyle=loos)
    if (basi[b].labloc[0]>-9999):
        ax1.text( basi[b].labloc[0],basi[b].labloc[1],basi[b].name , transform=data_crs, color='red' , bbox=bbox ) 
"""
ist=0
categories =[-2,0,1,2,3,4,5]  # -2 (sub-category storms) to 7 (Cat5)
cat_labels=['TD','TS','Cat 1','Cat 2','Cat 3','Cat 4','Cat 5',]
colors = ['beige','khaki','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['green','green','gold','orange', 'darkorange','red','red']  # Red for Cat5+



for ii in np.arange(len(yook[0])):
    ist=yook[0][ii]
    coo  = trx.categorize_storms(ibx.wind[ist, :], wind=True) 
    #ax1.scatter( ibx.lon[ist,:],ibx.lat[ist,:],transform=data_crs, s=.2 )

    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        mask = (coo == cat)
        ax1.scatter(ibx.lon[ist, mask], ibx.lat[ist, mask],
                    transform=data_crs, s=15., color=color ) #, label=f"{catlab}" if ii == 0 else "")





for ist in np.arange(len(tcMAAG)):
    windtc = np.array( tcMAAG[ist]['wind'] )
    lontc  = np.array( tcMAAG[ist]['lon'] )
    lattc  = np.array( tcMAAG[ist]['lat'] )
    coo  = trx.categorize_storms(  windtc , wind=True) 
    #ax1.scatter( ibx.lon[ist,:],ibx.lat[ist,:],transform=data_crs, s=.2 )


    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        mask = (coo == cat)
        ax1.scatter( lontc[mask]  , lattc[mask]  ,
                    transform=data_crs, s=0.4, color=color, label=f"{catlab}" if ist == 0 else "")

    #ax1.scatter( lontc  , lattc , transform=data_crs )

# Focus on Caribbean, for example
west, east, south, north = -120, -1, 5, 55
ax1.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

handles, labels = ax1.get_legend_handles_labels()
legend=ax1.legend(
    handles, 
    labels, 
    handler_map={plt.scatter: HandlerPathCollection()},
    loc='upper left', 
    title="Categories", 
    fontsize=14,  # Text size
    title_fontsize=16  # Title size
)
#for handle in legend.legendHandles:
for handle in legend.legend_handles:
    handle.set_sizes([100])  # Adjust this value for desired marker size



In [ ]:
from matplotlib.legend_handler import HandlerPathCollection

central_longitude=180.

#projPC = ccrs.Robinson(central_longitude=180.)
projPC = ccrs.PlateCarree(central_longitude=central_longitude) #180.)
data_crs = ccrs.PlateCarree()

basi = IBT.basin_outlines(Mediterranean=False,xNEPacific=True)
ipan=-1
"""
Add axes method 
ax = fig.add_axes([xmin , ymin , dx , dy , ])
"""
# Color mapping for categories
categories = np.arange(-2, 6)  # -2 (sub-category storms) to 7 (Cat5)
colors = ['gray', 'blue', 'cyan', 'green', 'yellow', 'orange', 'purple', 'red']  # Red for Cat5+


scale_fac = 1e-4 * 1.944**2 
denslv=21 #np.linspace(0,15,num=16)
#denslv=np.linspace(500,10_500,num=21)
denslv=np.linspace(0.2,4.0,num=20)

#scale_fac = 1.0 #1e-4 * 1.944**2 
#denslv= np.linspace(0.002,0.052,num=21) # - 0.05

nxplo=2
nyplo=1
xsize,ysize=0.7*20,0.7*15

fig = plt.figure(figsize=(nxplo*xsize, nyplo*ysize))
levs=[29,26,20,10]


n=1
Axes1 = Pu.axes_def(n=n,nxplo=nxplo,nyplo=nyplo, ypan=0.8 ) 
ax1 = fig.add_axes( Axes1 , projection=projPC )

ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
#ax1.set_title( "Spectrally-nudged CAM : Sept 15-30 2017" , fontsize=16)
ax1.set_title( "Spectrally-nudged CAM : Aug-Sept 2020" , fontsize=16)
ax1.add_feature(cfeature.LAND, facecolor='tan') 


ist=0
categories =[-2,0,1,2,3,4,5]  # -2 (sub-category storms) to 7 (Cat5)
cat_labels=['TD','TS','Cat 1','Cat 2','Cat 3','Cat 4','Cat 5',]
colors = ['beige','khaki','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['green','green','gold','orange', 'darkorange','red','red']  # Red for Cat5+



for ist in np.arange(len(tcMAAG)):
    windtc = np.array( tcMAAG[ist]['wind'] )
    lontc  = np.array( tcMAAG[ist]['lon'] )
    lattc  = np.array( tcMAAG[ist]['lat'] )
    coo  = trx.categorize_storms(  0.9*windtc , wind=True) 
    #ax1.scatter( ibx.lon[ist,:],ibx.lat[ist,:],transform=data_crs, s=.2 )


    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        mask = (coo == cat)
        ax1.scatter( lontc[mask]  , lattc[mask]  ,
                    transform=data_crs, s=5., color=color, label=f"{catlab}" if ist == 0 else "")

    #ax1.scatter( lontc  , lattc , transform=data_crs )

# Focus on Caribbean, for example
west, east, south, north = -120, -1, 5, 55
#ax1.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

handles, labels = ax1.get_legend_handles_labels()
legend=ax1.legend(
    handles, 
    labels, 
    handler_map={plt.scatter: HandlerPathCollection()},
    loc='upper left', 
    title="Categories", 
    fontsize=14,  # Text size
    title_fontsize=16  # Title size
)
#for handle in legend.legendHandles:
for handle in legend.legend_handles:
    handle.set_sizes([100])  # Adjust this value for desired marker size







########################################################################
n=2
Axes1 = Pu.axes_def(n=n,nxplo=nxplo,nyplo=nyplo, ypan=0.8 ) 
ax1 = fig.add_axes( Axes1 , projection=projPC )

ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
#ax1.set_title( "IBTrACS Aug-Sep 2017" , fontsize=16)
#ax1.set_title( "IBTrACS :  Sept 15-30 2017" , fontsize=16)
ax1.set_title( "IBTrACS :  Aug-Sept 2020" , fontsize=16)
ax1.add_feature(cfeature.LAND, facecolor='tan') 


ist=0
categories =[-2,0,1,2,3,4,5]  # -2 (sub-category storms) to 7 (Cat5)
cat_labels=['TD','TS','Cat 1','Cat 2','Cat 3','Cat 4','Cat 5',]
colors = ['beige','khaki','gold','orange', 'darkorange','red','red']  # Red for Cat5+
colors = ['green','green','gold','orange', 'darkorange','red','red']  # Red for Cat5+



for ii in np.arange(len(yook[0])):
    ist=yook[0][ii]
    coo  = trx.categorize_storms(ibx.wind[ist, :], wind=True) 
    #ax1.scatter( ibx.lon[ist,:],ibx.lat[ist,:],transform=data_crs, s=.2 )

    # Plot each category with its color
    for cat, color, catlab in zip(categories, colors,cat_labels):
        #mask = ((coo == cat) & (ibx.day[ist,:]>15)& (ibx.month[ist,:]==9))
        #mask = ((coo == cat) & (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        mask = ((coo == cat)) #& (ibx.month[ist,:]>=8) & (ibx.month[ist,:]<=9) )
        ax1.scatter(ibx.lon[ist, mask], ibx.lat[ist, mask],
                    transform=data_crs, s=5., color=color , label=f"{catlab}" if ii == 0 else "")





# Focus on Caribbean, for example
west, east, south, north = -120, -1, 5, 55
#ax1.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

handles, labels = ax1.get_legend_handles_labels()
legend=ax1.legend(
    handles, 
    labels, 
    handler_map={plt.scatter: HandlerPathCollection()},
    loc='upper left', 
    title="Categories", 
    fontsize=14,  # Text size
    title_fontsize=16  # Title size
)
#for handle in legend.legendHandles:
for handle in legend.legend_handles:
    handle.set_sizes([100])  # Adjust this value for desired marker size



In [ ]:
print( (1./2.)**0.11)